# Notebook 01 — Data audit and frozen LOEO protocol

**Project:** Robust Multisensor Banana Ripeness Classification across Independent Experiments: A Leakage-Safe Evaluation under Distribution Shift

Source dataset: University of Bath Research Data Archive, DOI `10.15125/BATH-01459`.

This notebook performs **data audit only**. It does not fit predictive models. The goal is to freeze provenance, reconstruct the inferred pre-balancing observations, build the master manifest, and define the nested Leave-One-Experiment-Out protocol before model selection.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import hashlib, zipfile, json, shutil, os

PROJECT = Path('/content/drive/MyDrive/Paper3_Banano/BANANA_RIPENESS_MULTISENSOR_Q1')
RAW = PROJECT/'01_RAW_ZIPS'
AUDIT = PROJECT/'03_DATA_AUDIT'
SPLITS = PROJECT/'04_FROZEN_SPLITS'
EXTRACTED_OUT = PROJECT/'02_EXTRACTED_DATA'
WORK = Path('/content/banana_ripeness_work')
WORK.mkdir(exist_ok=True)
for p in [RAW,AUDIT,SPLITS,EXTRACTED_OUT]: p.mkdir(parents=True,exist_ok=True)
print(PROJECT)

## 1. Freeze the four raw archives

In [ ]:
ZIP_NAMES = ['ds_34.zip','Spect.zip','TPHiS.zip','VOC.zip']

def digests(path):
    h256=hashlib.sha256(); hmd5=hashlib.md5()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''):
            h256.update(chunk); hmd5.update(chunk)
    return hmd5.hexdigest(), h256.hexdigest()

rows=[]
for name in ZIP_NAMES:
    p=RAW/name
    assert p.exists(), f'Missing {p}'
    md5, sha256=digests(p)
    with zipfile.ZipFile(p) as z: members=z.namelist()
    rows.append({'file':name,'bytes':p.stat().st_size,'md5':md5,'sha256':sha256,'zip_members':len(members),'members':' | '.join(members)})
checksums=pd.DataFrame(rows)
checksums.to_csv(AUDIT/'00_raw_zip_checksums.csv',index=False)
checksums

## 2. Extract to temporary Colab storage and verify CSV structure

In [ ]:
EX = WORK/'extracted'
if EX.exists(): shutil.rmtree(EX)
EX.mkdir()
for name in ZIP_NAMES:
    ds=name[:-4]
    out=EX/ds; out.mkdir()
    with zipfile.ZipFile(RAW/name) as z: z.extractall(out)
print('Extracted to',EX)

In [ ]:
COVERAGE={'ds_34':'Tests 3,4','Spect':'Tests 2,3,5','TPHiS':'Tests 2,3,4,5','VOC':'Tests 3,4'}
CUTOFF={'ds_34':10910,'Spect':11529,'TPHiS':17662,'VOC':10910}

def load_dataset(ds):
    parts=[]
    for split in ['train','test']:
        x=pd.read_csv(EX/ds/f'{ds}_x_{split}.csv').rename(columns={'Unnamed: 0':'source_index'})
        y=pd.read_csv(EX/ds/f'{ds}_y_{split}.csv').rename(columns={'Unnamed: 0':'source_index','0':'stage'})
        assert np.array_equal(x.source_index.to_numpy(),y.source_index.to_numpy())
        x['provided_split']=split; x['stage']=y.stage.to_numpy(); parts.append(x)
    return pd.concat(parts,ignore_index=True).sort_values('source_index').reset_index(drop=True)

DATA={ds:load_dataset(ds) for ds in ['ds_34','Spect','TPHiS','VOC']}
inv=[]
for ds,d in DATA.items():
    feats=[c for c in d.columns if c not in ['source_index','provided_split','stage']]
    inv.append({'dataset':ds,'coverage':COVERAGE[ds],'n_total':len(d),'n_train':(d.provided_split=='train').sum(),'n_test':(d.provided_split=='test').sum(),'n_predictors':len(feats),'n_missing':int(d[feats+['stage']].isna().sum().sum())})
inventory=pd.DataFrame(inv)
inventory.to_csv(AUDIT/'01_dataset_inventory.csv',index=False)
inventory

## 3. Reconstruct the inferred pre-balancing block

The released files are balanced. In each dataset, the low-index contiguous block reproduces the recorded unbalanced observations; the subsequent tail fills minority classes exactly to the balanced class count. We therefore store this as an **inference** (`synthetic_tail_inferred`), not as an author-provided field.

In [ ]:
audit=[]
for ds,d in DATA.items():
    orig=d[d.source_index<CUTOFF[ds]]; tail=d[d.source_index>=CUTOFF[ds]]
    for stage in range(1,6):
        audit.append({'dataset':ds,'stage':stage,'original_inferred':int((orig.stage==stage).sum()),'synthetic_tail_inferred':int((tail.stage==stage).sum()),'balanced_total':int((d.stage==stage).sum())})
synthetic_audit=pd.DataFrame(audit)
synthetic_audit.to_csv(AUDIT/'02_original_vs_synthetic_audit.csv',index=False)
synthetic_audit

## 4. Build common original-data manifest and reconstruct experiment IDs

In [ ]:
T=DATA['TPHiS']
T=T[T.source_index<CUTOFF['TPHiS']].copy().sort_values('source_index')
T['test_id']=pd.Series(np.select([T.source_index.between(0,1719),T.source_index.between(1720,6496),T.source_index.between(6497,12629),T.source_index.between(12630,17661)],[2,3,4,5],default=np.nan),index=T.index,dtype='Int64')
T['test5_stream']=pd.Series(pd.NA,index=T.index,dtype='Int64')
T.loc[T.source_index.between(12630,15145),'test5_stream']=1
T.loc[T.source_index.between(15146,17661),'test5_stream']=2
T['master_id']=T.source_index.astype(int)
T['is_synthetic_inferred']=False
T['has_tph']=True; T['has_spectral']=True; T['has_voc']=T.test_id.isin([3,4]); T['has_external_env']=T.test_id.isin([3,4])

# map companion dataset indices
T['spect_source_index']=pd.Series(pd.NA,index=T.index,dtype='Int64')
T.loc[T.test_id.isin([2,3]),'spect_source_index']=T.loc[T.test_id.isin([2,3]),'source_index'].astype('Int64')
T.loc[T.test_id.eq(5),'spect_source_index']=(T.loc[T.test_id.eq(5),'source_index']-6133).astype('Int64')
T['ds34_source_index']=pd.Series(pd.NA,index=T.index,dtype='Int64')
T.loc[T.test_id.isin([3,4]),'ds34_source_index']=(T.loc[T.test_id.isin([3,4]),'source_index']-1720).astype('Int64')
T['voc_source_index']=T.ds34_source_index.copy()

manifest=T[['master_id','test_id','test5_stream','stage','is_synthetic_inferred','source_index','spect_source_index','ds34_source_index','voc_source_index','provided_split','has_tph','has_spectral','has_voc','has_external_env']].rename(columns={'source_index':'tphis_source_index','provided_split':'provided_split_tphis'})
manifest.to_csv(AUDIT/'05_master_manifest_original.csv',index=False)
manifest.groupby(['test_id','test5_stream'],dropna=False).stage.value_counts().rename('n').reset_index()

## 5. Freeze nested Leave-One-Experiment-Out assignments

In [ ]:
outer=[]
for ext in [2,3,4,5]:
    f=manifest[['master_id','test_id','test5_stream','stage']].copy()
    f['outer_test_id']=ext
    f['outer_role']=np.where(f.test_id.eq(ext),'test','development')
    f.to_csv(SPLITS/f'outer_test_{ext}.csv',index=False)
    outer.append(f)
pd.concat(outer,ignore_index=True).to_csv(SPLITS/'outer_loeo_assignments_long.csv',index=False)
print('Frozen LOEO files written to',SPLITS)

## Frozen rules

1. Main confirmatory analysis uses inferred-original observations only.
2. External experiment is untouched until final prediction.
3. Scaling, PCA, UMAP, resampling, feature selection, tuning and calibration are learned only from development data.
4. SMOTE, if evaluated, occurs only inside inner-training folds.
5. Both Test-5 streams remain together as one experiment.
6. No model is selected using outer-test performance.

**Next notebook:** PCA/UMAP and quantitative domain-shift diagnostics, followed by nested-LOEO ML benchmarking.